In [0]:
dbutils.widgets.text("ClientContainer", "claimsprocessing", "Client Container")
dbutils.widgets.text("SubGroupConfigPath", "", "SubGroup Config Path")

widget_client = dbutils.widgets.get("ClientContainer").strip()
widget_config = dbutils.widgets.get("SubGroupConfigPath").strip()

clientContainer = widget_client if widget_client else "claimsprocessing"
subGroupConfigPath = widget_config

safe_catalog = f"{clientContainer}"
print(f"Client Container: {clientContainer}")
print(f"Config Path: {subGroupConfigPath}")

In [0]:
from pyspark.sql.functions import explode, col
import os, json

def path_exists(path):
    try:
        if '.' in path and not path.startswith('/'):
            return spark.catalog.tableExists(path)
        if path.startswith('/'):
            try:
                dbutils.fs.ls(path)
                return True
            except:
                return False
        return spark.catalog.tableExists(path)
    except:
        return False

In [0]:
print(f"Reading config from: {subGroupConfigPath}")
with open(subGroupConfigPath, 'r') as f:
    config_data = json.load(f)

subLayerProcessing = config_data.get("SubLayerProcessing", [])
processing_results = []

for entity_row in subLayerProcessing:
    subGroupEntity = entity_row.get("SubGroupEntity", "Unknown")
    print(f"\n{'='*60}")
    print(f"Processing Entity: {subGroupEntity}")
    print(f"{'='*60}")
    
    all_sources_valid = True
    destinationTable = entity_row.get("DestinationTable", "").replace("#clientCode", safe_catalog)
    print(f"Destination: {destinationTable}")
    
    dest_exists = path_exists(destinationTable)
    source_tables = entity_row.get("SourceTables", [])
    
    for source_row in source_tables:
        entity_name = source_row.get("Entity", "")
        source_table = source_row.get("SourceTable", "").replace("#clientCode", safe_catalog)
        source_format = source_row.get("SourceFormat", "delta")
        
        print(f"  Loading source: {entity_name} from {source_table}")
        try:
            if path_exists(source_table):
                if '.' in source_table and not source_table.startswith('/'):
                    df_file = spark.table(source_table)
                else:
                    df_file = spark.read.format(source_format).option("header", "true").option("inferSchema", "true").load(source_table)
                df_file.createOrReplaceTempView(entity_name)
                print(f"  {entity_name} temp view created")
            else:
                all_sources_valid = False
                print(f"  Source path/table does not exist: {source_table}")
        except Exception as e:
            all_sources_valid = False
            print(f"  Failed to load {entity_name}: {str(e)}")
    
    if all_sources_valid:
        try:
            # Load SQL Script (Inline or External Path)
            if "SQLScriptPath" in entity_row:
                sql_path_rel = entity_row.get("SQLScriptPath", "")
                gold_dir = os.path.dirname(os.path.dirname(subGroupConfigPath))
                sql_path_abs = os.path.normpath(os.path.join(gold_dir, sql_path_rel))
                with open(sql_path_abs, "r", encoding="utf-8") as sf:
                    mainSQLQuery = sf.read().replace("#clientCode", safe_catalog)
            else:
                mainSQLQuery = entity_row.get("SQLScript", "").replace("#clientCode", safe_catalog)
                
            mDF = spark.sql(mainSQLQuery)
            mDF.createOrReplaceTempView("tempSQLScript")
            print(f"  tempSQLScript view created with {mDF.count()} rows")
            
            # Ensure target table exists
            if not dest_exists:
                table_parts = destinationTable.split('.')
                if len(table_parts) == 3:
                    spark.sql(f"CREATE DATABASE IF NOT EXISTS {table_parts[0]}.{table_parts[1]}")
                mDF.limit(0).write.format("delta").mode("overwrite").saveAsTable(destinationTable)
                dest_exists = True
            
            spark.table(destinationTable).createOrReplaceTempView("DestinationTable")
            
            # Load Merge Script (Inline or External Path)
            if "MergeScriptPath" in entity_row:
                merge_path_rel = entity_row.get("MergeScriptPath", "")
                gold_dir = os.path.dirname(os.path.dirname(subGroupConfigPath))
                merge_path_abs = os.path.normpath(os.path.join(gold_dir, merge_path_rel))
                with open(merge_path_abs, "r", encoding="utf-8") as mf:
                    mergeScript = mf.read().replace("DestinationTable", destinationTable)
            else:
                mergeScript = entity_row.get("MergeScript", "").replace("DestinationTable", destinationTable)
                
            spark.sql(mergeScript)
            print(f"  MERGE completed successfully for {subGroupEntity}")
            processing_results.append({"entity": subGroupEntity, "status": "SUCCESS"})
        except Exception as e:
            print(f"  Failed execution for {subGroupEntity}: {str(e)}")
            processing_results.append({"entity": subGroupEntity, "status": "FAILED", "error": str(e)})

return_message = f"Processed {len(processing_results)} entities"
print(return_message)
dbutils.notebook.exit(return_message)